# Maximum Likelihood Estimation (MLE): Normal and Bernoulli
Works on **any dataset** (CSV / Excel). A file explorer opens, then you choose a numeric column for the **Normal** MLE and a binary column for the **Bernoulli** MLE.

**Idea:** choose the parameter value that makes the observed data most probable.

$$L(\theta) = \prod_{i=1}^n f(x_i;\theta), \qquad \ell(\theta) = \sum_{i=1}^n \ln f(x_i;\theta), \qquad \hat\theta = \arg\max_\theta \ell(\theta)$$

| Distribution | MLE |
|---|---|
| Normal$(\mu,\sigma^2)$ | $\hat\mu = \bar{x}$, $\quad\hat\sigma^2 = \frac{1}{n}\sum (x_i-\bar{x})^2$ |
| Bernoulli$(p)$ | $\hat p = \frac{1}{n}\sum x_i = \bar{x}$ |

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.optimize import minimize

np.random.seed(42)

## 2. Select Dataset from File Explorer

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
file_path = filedialog.askopenfilename(
    title="Select your dataset",
    filetypes=[("Data files", "*.csv *.xlsx *.xls *.txt *.tsv"), ("All files", "*.*")]
)
root.destroy()

if not file_path:
    raise SystemExit("No file selected. Re-run this cell.")

print("Selected file:", file_path)

if file_path.endswith('.csv'):
    df = pd.read_csv(file_path)
elif file_path.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_path)
else:
    df = pd.read_csv(file_path, sep=None, engine='python')

print("Shape:", df.shape)
df.head()

# Part A: MLE for the Normal Distribution

## 3. Choose a Numeric Column

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:")
for i, c in enumerate(num_cols):
    print(f"{i}: {c}")

col = input("\nEnter column for NORMAL MLE (name or index): ").strip()
if col.isdigit() and col not in df.columns:
    col = num_cols[int(col)]

x = df[col].dropna().values.astype(float)
n = len(x)
print(f"\nColumn: {col},  n = {n}")

## 4. Check if Normal is a Reasonable Model

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x, kde=True, stat='density', ax=ax[0], color='skyblue')
ax[0].set_title(f'Histogram of {col}')
stats.probplot(x, dist='norm', plot=ax[1])
plt.tight_layout(); plt.show()

sub = x if n <= 5000 else np.random.choice(x, 5000, replace=False)
print(f"Skewness = {stats.skew(x):.3f}, Excess kurtosis = {stats.kurtosis(x):.3f}")
print(f"Shapiro-Wilk p-value = {stats.shapiro(sub)[1]:.4g}  (p > 0.05 supports normality)")

## 5. Analytical MLE (closed form)
Log-likelihood: $\ell(\mu,\sigma^2) = -\frac{n}{2}\ln(2\pi) - \frac{n}{2}\ln\sigma^2 - \frac{1}{2\sigma^2}\sum(x_i-\mu)^2$

Setting derivatives to zero gives $\hat\mu=\bar{x}$ and $\hat\sigma^2 = \frac1n\sum(x_i-\bar{x})^2$.

In [ ]:
mu_mle = x.mean()
var_mle = np.mean((x - mu_mle)**2)          # divides by n  (MLE, biased)
sigma_mle = np.sqrt(var_mle)
var_unbiased = x.var(ddof=1)                # divides by n-1 (unbiased)

print(f"MLE of mu       = {mu_mle:.6f}")
print(f"MLE of sigma^2  = {var_mle:.6f}   (divides by n)")
print(f"MLE of sigma    = {sigma_mle:.6f}")
print(f"Unbiased s^2    = {var_unbiased:.6f}   (divides by n-1)")

## 6. Log-Likelihood Function

In [ ]:
def normal_loglik(params, data):
    mu, sigma = params
    if sigma <= 0:
        return -np.inf
    m = len(data)
    return -m/2*np.log(2*np.pi) - m*np.log(sigma) - np.sum((data - mu)**2) / (2*sigma**2)

ll_max = normal_loglik([mu_mle, sigma_mle], x)
print(f"Log-likelihood at MLE = {ll_max:.4f}")

# Check using scipy
print(f"Check with scipy      = {np.sum(stats.norm.logpdf(x, mu_mle, sigma_mle)):.4f}")

## 7. Numerical MLE (minimize negative log-likelihood)
This is how MLE is found when no closed form exists.

In [ ]:
def neg_loglik(params, data):
    mu, log_sigma = params           # optimise log(sigma) so sigma stays positive
    return -normal_loglik([mu, np.exp(log_sigma)], data)

start = [np.median(x), np.log(x.std() + 1e-9)]
res = minimize(neg_loglik, start, args=(x,), method='Nelder-Mead')

mu_num, sigma_num = res.x[0], np.exp(res.x[1])
print("Optimizer converged:", res.success)
print(f"Numerical MLE: mu = {mu_num:.6f}, sigma = {sigma_num:.6f}")
print(f"Analytical MLE: mu = {mu_mle:.6f}, sigma = {sigma_mle:.6f}")

# scipy built-in
mu_sp, sigma_sp = stats.norm.fit(x)
print(f"scipy norm.fit: mu = {mu_sp:.6f}, sigma = {sigma_sp:.6f}")

## 8. Visualise the Likelihood

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

# (a) profile over mu (sigma fixed at MLE)
mus = np.linspace(mu_mle - 3*sigma_mle/np.sqrt(n)*3, mu_mle + 3*sigma_mle/np.sqrt(n)*3, 300)
ax[0].plot(mus, [normal_loglik([m, sigma_mle], x) for m in mus])
ax[0].axvline(mu_mle, color='r', ls='--', label=f'MLE = {mu_mle:.3f}')
ax[0].set_title('Log-likelihood vs mu (sigma fixed)'); ax[0].set_xlabel('mu'); ax[0].legend()

# (b) profile over sigma (mu fixed at MLE)
sigs = np.linspace(sigma_mle*0.6, sigma_mle*1.6, 300)
ax[1].plot(sigs, [normal_loglik([mu_mle, s], x) for s in sigs])
ax[1].axvline(sigma_mle, color='r', ls='--', label=f'MLE = {sigma_mle:.3f}')
ax[1].set_title('Log-likelihood vs sigma (mu fixed)'); ax[1].set_xlabel('sigma'); ax[1].legend()

# (c) contour over (mu, sigma)
M, S = np.meshgrid(np.linspace(mus.min(), mus.max(), 120), np.linspace(sigs.min(), sigs.max(), 120))
Z = np.array([[normal_loglik([m, s], x) for m, s in zip(rm, rs)] for rm, rs in zip(M, S)])
cs = ax[2].contourf(M, S, Z, levels=30, cmap='viridis')
ax[2].plot(mu_mle, sigma_mle, 'r*', ms=16, label='MLE')
ax[2].set_title('Log-likelihood surface'); ax[2].set_xlabel('mu'); ax[2].set_ylabel('sigma'); ax[2].legend()
plt.colorbar(cs, ax=ax[2])
plt.tight_layout(); plt.show()

## 9. Fitted Normal vs Data

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.histplot(x, stat='density', bins=30, color='lightgray', label='Data')
xs = np.linspace(x.min(), x.max(), 400)
plt.plot(xs, stats.norm.pdf(xs, mu_mle, sigma_mle), 'r-', lw=2.5,
         label=f'Normal MLE (mu={mu_mle:.2f}, sigma={sigma_mle:.2f})')
plt.legend(); plt.title(f'Normal distribution fitted by MLE to {col}'); plt.show()

## 10. Standard Errors and Confidence Intervals
Fisher information gives $SE(\hat\mu)=\hat\sigma/\sqrt n$ and $SE(\hat\sigma^2)=\hat\sigma^2\sqrt{2/n}$ (asymptotically).

In [ ]:
se_mu = sigma_mle / np.sqrt(n)
se_var = var_mle * np.sqrt(2 / n)
z = 1.96

print(f"mu      = {mu_mle:.4f},  SE = {se_mu:.4f},  95% CI = ({mu_mle - z*se_mu:.4f}, {mu_mle + z*se_mu:.4f})")
print(f"sigma^2 = {var_mle:.4f},  SE = {se_var:.4f},  95% CI = ({var_mle - z*se_var:.4f}, {var_mle + z*se_var:.4f})")

# Numerical check: SE from the Hessian of the negative log-likelihood (observed Fisher information)
from scipy.optimize import approx_fprime
def hessian(f, p, eps=1e-4):
    k = len(p); H = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            pp = np.array(p, float)
            def g(a, b):
                q = pp.copy(); q[i] += a; q[j] += b; return f(q)
            H[i, j] = (g(eps, eps) - g(eps, -eps) - g(-eps, eps) + g(-eps, -eps)) / (4*eps**2)
    return H

H = hessian(lambda p: -normal_loglik(p, x), [mu_mle, sigma_mle])
cov = np.linalg.inv(H)
print("\nSE from Hessian -> mu: %.4f, sigma: %.4f" % tuple(np.sqrt(np.diag(cov))))

## 11. Bias of the MLE of Variance (simulation)
$E[\hat\sigma^2_{MLE}] = \frac{n-1}{n}\sigma^2$, so the MLE underestimates $\sigma^2$ for small n, but is consistent.

In [ ]:
true_mu, true_sigma = 10, 3
sample_sizes = [5, 10, 20, 50, 100, 500]
reps = 5000
rows = []
for m in sample_sizes:
    mle_vars = np.array([np.var(np.random.normal(true_mu, true_sigma, m), ddof=0) for _ in range(reps)])
    unb_vars = mle_vars * m / (m - 1)
    rows.append({'n': m, 'True sigma^2': true_sigma**2,
                 'Mean MLE var': mle_vars.mean(), 'Mean unbiased var': unb_vars.mean(),
                 'Theory (n-1)/n*sigma^2': (m-1)/m*true_sigma**2})
display(pd.DataFrame(rows).set_index('n').round(3))

# Part B: MLE for the Bernoulli Distribution

## 12. Choose a Binary Column (or create one)

In [ ]:
print("Columns:")
for i, c in enumerate(df.columns):
    print(f"{i}: {c}  ({df[c].nunique()} unique values)")

bcol = input("\nEnter column for BERNOULLI MLE (name or index): ").strip()
if bcol.isdigit() and bcol not in df.columns:
    bcol = df.columns[int(bcol)]

series = df[bcol].dropna()
uniq = series.unique()

if len(uniq) == 2:
    print("Binary column with values:", uniq)
    success = input(f"Which value counts as SUCCESS (1)? ").strip()
    # match the typed text with actual values
    success_val = [u for u in uniq if str(u) == success]
    if not success_val:
        raise ValueError("Value not found. Re-run this cell and type one of: " + str(list(uniq)))
    b = (series == success_val[0]).astype(int).values
else:
    print("Not binary -> convert using a threshold on a numeric column.")
    thr = float(input("Enter threshold (success if value > threshold): "))
    b = (pd.to_numeric(series, errors='coerce') > thr).astype(int).values

nb = len(b)
k = b.sum()
print(f"\nn = {nb}, successes = {k}, failures = {nb - k}")

## 13. Analytical MLE
$\ell(p) = k\ln p + (n-k)\ln(1-p)$, setting $\ell'(p)=0$ gives $\hat p = k/n$.

In [ ]:
p_mle = k / nb
print(f"MLE of p = k/n = {k}/{nb} = {p_mle:.6f}")

def bern_loglik(p, data):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return np.sum(data) * np.log(p) + (len(data) - np.sum(data)) * np.log(1 - p)

print(f"Log-likelihood at MLE = {bern_loglik(p_mle, b):.4f}")
print(f"Check with scipy      = {np.sum(stats.bernoulli.logpmf(b, p_mle)):.4f}")

## 14. Numerical MLE and Grid Search

In [ ]:
# (a) Numerical optimisation
res_b = minimize(lambda p: -bern_loglik(p[0], b), x0=[0.5], bounds=[(1e-6, 1 - 1e-6)], method='L-BFGS-B')
print(f"Numerical MLE of p = {res_b.x[0]:.6f}")

# (b) Grid search
grid = np.linspace(0.001, 0.999, 999)
ll = np.array([bern_loglik(p, b) for p in grid])
print(f"Grid-search MLE     = {grid[ll.argmax()]:.3f}")

## 15. Visualise Likelihood

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))

lik = np.exp(ll - ll.max())     # scaled likelihood (avoids underflow)
ax[0].plot(grid, lik)
ax[0].axvline(p_mle, color='r', ls='--', label=f'MLE = {p_mle:.3f}')
ax[0].set_title('Likelihood L(p) (scaled)'); ax[0].set_xlabel('p'); ax[0].legend()

ax[1].plot(grid, ll)
ax[1].axvline(p_mle, color='r', ls='--', label=f'MLE = {p_mle:.3f}')
ax[1].set_title('Log-likelihood l(p)'); ax[1].set_xlabel('p'); ax[1].legend()

ax[2].bar(['Failure (0)', 'Success (1)'], [nb - k, k], color=['lightcoral', 'lightgreen'])
ax[2].set_title(f'Observed counts of {bcol}')
plt.tight_layout(); plt.show()

## 16. Standard Error and Confidence Intervals
Fisher information $I(p) = \frac{n}{p(1-p)}$ so $SE(\hat p)=\sqrt{\hat p(1-\hat p)/n}$.

In [ ]:
se_p = np.sqrt(p_mle * (1 - p_mle) / nb)
print(f"SE(p_hat) = {se_p:.5f}")
print(f"Wald 95% CI = ({p_mle - 1.96*se_p:.4f}, {p_mle + 1.96*se_p:.4f})")

# Wilson interval is better for small n or p near 0/1
wilson = stats.binomtest(int(k), nb).proportion_ci(confidence_level=0.95, method='wilson')
print(f"Wilson 95% CI = ({wilson.low:.4f}, {wilson.high:.4f})")

# Likelihood-ratio CI: p where 2*(l(p_hat) - l(p)) <= 3.84
inside = grid[2*(ll.max() - ll) <= stats.chi2.ppf(0.95, 1)]
print(f"Likelihood-ratio 95% CI = ({inside.min():.4f}, {inside.max():.4f})")

## 17. Consistency of the Bernoulli MLE (simulation)

In [ ]:
true_p = p_mle if 0 < p_mle < 1 else 0.3
ns = [5, 10, 20, 50, 100, 500, 1000, 5000]
est_mean, est_sd = [], []
for m in ns:
    ests = np.random.binomial(m, true_p, 3000) / m
    est_mean.append(ests.mean()); est_sd.append(ests.std())

plt.figure(figsize=(8, 4.5))
plt.errorbar(ns, est_mean, yerr=est_sd, fmt='o-', capsize=4)
plt.axhline(true_p, color='r', ls='--', label=f'true p = {true_p:.3f}')
plt.xscale('log'); plt.xlabel('sample size n'); plt.ylabel('p_hat')
plt.title('MLE converges to true p as n grows (consistency)'); plt.legend(); plt.show()

## 18. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Distribution': ['Normal', 'Normal', 'Bernoulli'],
    'Parameter': [f'mu ({col})', f'sigma^2 ({col})', f'p ({bcol})'],
    'MLE': [mu_mle, var_mle, p_mle],
    'Standard Error': [se_mu, se_var, se_p],
    'Formula': ['mean(x)', 'mean((x - mean)^2)', 'k / n']
})
summary.round(5)

## Key Points (for viva)
- **Likelihood** $L(\theta)$: probability of the observed data viewed as a function of the parameter. We maximise $\ln L$ because it turns products into sums and has the same maximiser.
- **Steps:** write likelihood, take log, differentiate, set to 0, solve, check second derivative is negative.
- **Normal:** $\hat\mu=\bar{x}$; $\hat\sigma^2=\frac1n\sum(x_i-\bar{x})^2$ is **biased** (expectation $\frac{n-1}{n}\sigma^2$) but **consistent**. The unbiased version divides by $n-1$.
- **Bernoulli:** $\hat p = \bar{x}$ = sample proportion, which is unbiased.
- **Properties of MLE:** consistent, asymptotically normal, asymptotically efficient (reaches the Cramer-Rao bound), and invariant (MLE of $g(\theta)$ is $g(\hat\theta)$).
- **Fisher information:** Normal mean $I(\mu)=n/\sigma^2$; Bernoulli $I(p)=n/(p(1-p))$. $SE = 1/\sqrt{I(\hat\theta)}$.
- **MLE vs Method of Moments:** for these distributions they give the same answer. MLE is generally more efficient.
- If all observations are 0 or all 1, then $\hat p$ is 0 or 1 (boundary), a known weakness of MLE for small samples.